In [ ]:
! pip install semantic-link-labs --q


In [ ]:
from sempy_labs import admin
import sempy_labs._icons as icons
import pandas as pd
from uuid import UUID
from typing import Optional, List


In [ ]:
def delete_all_capacity_tenant_setting_override(
    capacity: Optional[str | UUID] = None,
    tenant_setting: Optional[str] = None,
    dry_run: bool = True
) -> pd.DataFrame:
    """
    Deletes and returns list of tenant setting overrides that override at the capacities after applying the tenant_setting filter.

    This is a wrapper function for the following APIs: `Tenants - List Capacities Tenant Settings Overrides <https://learn.microsoft.com/rest/api/fabric/admin/tenants/list-capacities-tenant-settings-overrides>`_
    and `Tenants - Delete Capacity Tenant Setting Override <https://learn.microsoft.com/rest/api/fabric/admin/tenants/delete-capacity-tenant-setting-override>`_.

    Service Principal Authentication is supported (see `here <https://github.com/microsoft/semantic-link-labs/blob/main/notebooks/Service%20Principal.ipynb>`_ for examples).

    Parameters
    ----------
    capacity : str | uuid.UUID, default = None
        The capacity name or ID.
        Defaults to None which resolves to showing/deleting all capacities.
    tenant_setting : str, default = None
        The tenant setting name. Example: "TenantSettingForCapacityDelegatedSwitch"
        Defaults to None which resolves to showing/deleting all tenant settings.
    dry_run : bool, default = True
        Show or delete the tenant settings override at the capacities
        Defaults to True which resolves to showing the tenant settings override at the capacities.

    Returns
    -------
    pandas.DataFrame
        A pandas dataframe showing a list of tenant setting overrides that override at the capacities after applying the tenant_setting filter.
    """

    df = (
        admin.list_capacity_tenant_settings_overrides(
            capacity = capacity,
            return_dataframe = True
        )
    )

    # Filter tenant_setting
    if tenant_setting is None:
        df_filt = df
    else:
        df_filt = df[
            df["Setting Name"] == tenant_setting
        ]

    if df_filt.empty:
        print(f"{icons.yellow_dot} No rows found for the selected parameters: '{tenant_setting}' tenant setting in '{capacity}' capacity.")
    else:
        for _, row in df_filt.iterrows():
            capacity = row["Capacity Id"]
            tenant_setting = row["Setting Name"]

            if dry_run:
                print(f"{icons.yellow_dot} The '{tenant_setting}' tenant setting will be removed from the '{capacity}' capacity.")
            else:
                try:
                    admin.delete_capacity_tenant_setting_override(capacity, tenant_setting)
                except Exception as e:
                    print(f"{icons.red_dot} Error deleting override (Capacity={capacity}, Setting={tenant_setting}): {e}")

    return df_filt


In [ ]:
capacity = None #"<Capacity Name or Capacity ID>"
tenant_setting = None #"<Tenant Setting>" e.g.: "ArtifactGraphPreview"
dry_run = True


In [ ]:
pd_capacity_settings_overrides_returned = (
    delete_all_capacity_tenant_setting_override( # To use the function defined above.
    # admin.delete_all_capacity_tenant_setting_override( # To use the Semantic Link Labs function if accepted.
        capacity = capacity,
        tenant_setting = tenant_setting,
        dry_run = dry_run
    )
)


In [ ]:
display (
    pd_capacity_settings_overrides_returned
)
